# Task 4 - Transfer Learning and Generalization

**Goal:** fine-tune ResNet-152 on CIFAR-10 (a dataset different from ImageNet) and answer two questions:

- **Pre-trained vs random init:** does starting from ImageNet weights beat training from random initialization?
- **How much to fine-tune:** head only, last block only, or the full backbone?

Note: assignment items (d) and (e) restate (b) and (c); the matrix below covers both comparisons. Runs 2 vs 3 answer *"final block vs full backbone"*; runs 3 vs 4 answer *"pre-trained vs random"*.

## Experiment matrix

| Run                     | Weights  | Trainable         | LR    |
|-------------------------|----------|-------------------|-------|
| 1. baseline (Task 1)    | ImageNet | head only         | 1e-3  |
| 2. fine-tune block 4    | ImageNet | layer4 + head     | 1e-4  |
| 3. fine-tune full       | ImageNet | all               | 1e-4  |
| 4. from scratch         | random   | all               | 1e-4  |

The backbone is fine-tuned at a lower LR than the head - standard practice, since large steps destroy pre-trained features. All runs use Adam, 5 epochs, and identical seeds (same head init and data order), so the only differences are weights and trainable scope. Each run is timed; the wall-clock time is part of the comparison.

## 1. Environment & Data

In [ ]:
%pip install torch torchvision torchaudio scikit-learn

In [ ]:
# ---------- Imports ----------
import time
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
from torchvision.models import resnet152, ResNet152_Weights

# ---------- Configuration ----------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 64
num_epochs = 5

print(torch.__version__)
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
print(f"Using: {device}")

In [ ]:
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

In [ ]:
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
train_loader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)

valset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_val)
val_loader = torch.utils.data.DataLoader(valset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train images: {len(trainset)} ({len(train_loader)} batches of {batch_size})")
print(f"Val images:   {len(valset)} ({len(val_loader)} batches of {batch_size})")

## 2. Training function (all four runs)

In [ ]:
def run_transfer(name, init="imagenet", trainable="head", lr=1e-3, epochs=num_epochs, seed=0):
    """Train ResNet-152 on CIFAR-10 under a given setting.

    init       : "imagenet" (pre-trained weights) or "random" (from scratch)
    trainable  : "head"    -> only the 10-class head trains
                 "block4"  -> layer4 + head trains, rest frozen
                 "full"    -> the whole network trains
    """
    torch.manual_seed(seed)  # identical head init across runs

    if init == "imagenet":
        model = resnet152(weights=ResNet152_Weights.DEFAULT)
    else:
        model = resnet152(weights=None)  # random initialization (Kaiming default)

    if trainable in ("head", "block4"):
        for param in model.parameters():
            param.requires_grad = False
        if trainable == "block4":
            for param in model.layer4.parameters():
                param.requires_grad = True

    model.fc = nn.Linear(model.fc.in_features, 10)
    model = model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=lr)
    torch.manual_seed(seed)  # identical data order across runs

    history = {"name": name, "init": init, "trainable": trainable, "lr": lr,
               "train_loss": [], "train_acc": [], "val_loss": [], "val_acc": [], "time_s": 0.0}

    t0 = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for i, (images, labels) in enumerate(train_loader):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)
            _, pred = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (pred == labels).sum().item()
            if (i + 1) % 200 == 0:
                print(f"[{name}] Epoch {epoch+1}/{epochs} | Step {i+1}/{len(train_loader)} | Loss {loss.item():.4f}")

        tl, ta = running_loss / total, 100.0 * correct / total

        model.eval()
        vloss, vc, vt = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                vloss += loss.item() * images.size(0)
                _, pred = torch.max(outputs, 1)
                vt += labels.size(0)
                vc += (pred == labels).sum().item()
        vl, va = vloss / vt, 100.0 * vc / vt

        history["train_loss"].append(tl); history["train_acc"].append(ta)
        history["val_loss"].append(vl); history["val_acc"].append(va)
        print(f"[{name}] Epoch {epoch+1}/{epochs} | Train Loss {tl:.4f} | Train Acc {ta:.2f}% | Val Loss {vl:.4f} | Val Acc {va:.2f}%")

    history["time_s"] = time.perf_counter() - t0
    print(f"[{name}] finished in {history['time_s']:.1f} s")
    return history

## 3. Run 1 - baseline: pre-trained, head only (reference from Task 1)

In [ ]:
# Recorded in Task 1 (same protocol: ImageNet init, head only, Adam lr=1e-3, 5 epochs).
hist_baseline = {"name": "baseline (head only)", "init": "ImageNet", "trainable": "head only", "lr": 1e-3,
                 "train_loss": [0.6985, 0.4893, 0.4499, 0.4225, 0.4062],
                 "train_acc": [78.52, 83.64, 84.87, 85.82, 86.18],
                 "val_loss": [0.5102, 0.4712, 0.4560, 0.4608, 0.4455],
                 "val_acc": [83.47, 84.56, 84.71, 84.83, 85.21], "time_s": 0.0}

RERUN_BASELINE = False  # set True to re-train the baseline here instead
if RERUN_BASELINE:
    hist_baseline = run_transfer("baseline (head only)", init="imagenet", trainable="head", lr=1e-3)
print("Baseline history ready.")

## 4. Run 2 - fine-tune only the last block (layer4 + head)

In [ ]:
hist_block4 = run_transfer("fine-tune block 4", init="imagenet", trainable="block4", lr=1e-4)

## 5. Run 3 - fine-tune the full backbone (pre-trained weights)

In [ ]:
hist_full = run_transfer("fine-tune full", init="imagenet", trainable="full", lr=1e-4)

## 6. Run 4 - training from random initialization (full backbone)

In [ ]:
hist_scratch = run_transfer("from scratch", init="random", trainable="full", lr=1e-4)

## 7. Comparison

In [ ]:
histories = [hist_baseline, hist_block4, hist_full, hist_scratch]

print(f"{'Run':<28}{'Init':<9}{'Trainable':<14}{'Final Val Acc':>14}{'Time':>9}")
print("-" * 74)
for h in histories:
    print(f"{h['name']:<28}{h['init']:<9}{h['trainable']:<14}{h['val_acc'][-1]:>12.2f}%{h['time_s']:>8.0f}s")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
epochs = range(1, num_epochs + 1)
markers = ["o", "s", "^", "D"]

# Convergence curves
for h, m in zip(histories, markers):
    axes[0].plot(epochs, h["val_acc"], marker=m, label=h["name"])
axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Validation accuracy (%)")
axes[0].set_title("Convergence of the four settings")
axes[0].grid(alpha=0.3); axes[0].legend(fontsize=8)

# Final accuracy vs compute
labels = [h["name"].replace("fine-tune ", "FT ") for h in histories]
accs = [h["val_acc"][-1] for h in histories]
times = [h["time_s"] for h in histories]
bars = axes[1].bar(labels, accs, color=["tab:blue", "tab:orange", "tab:green", "tab:red"], alpha=0.85)
for b, h in zip(bars, histories):
    axes[1].text(b.get_x() + b.get_width() / 2, b.get_height() + 0.5,
                 f"{h['val_acc'][-1]:.1f}%\n{h['time_s']/60:.1f} min", ha="center", fontsize=8)
axes[1].set_ylabel("Final validation accuracy (%)")
axes[1].set_title("Accuracy vs training time")
axes[1].set_ylim(0, 100)
axes[1].tick_params(axis="x", rotation=12)

plt.tight_layout()
plt.savefig("transfer_comparison.png", dpi=150)
plt.show()

## 8. Discussion

### Which setting provides the best trade-off between compute and accuracy?

- **Head only (baseline)** is by far the cheapest (only ~0.02 M trainable parameters) and already reaches ~85% - because the frozen ImageNet features are directly usable. But it caps the achievable accuracy.
- **Fine-tuning block 4** adds most of the accuracy gain for a small fraction of the full-training cost: it adapts exactly the layers that are most task-specific.
- **Full fine-tuning** gives the highest accuracy (the whole hierarchy gets adapted to CIFAR-10) but costs several times more compute (every epoch now back-propagates through ~60 M parameters).
- **From scratch** spends the same compute as run 3 but starts from random features and simply does not converge in 5 epochs - it ends far below every pre-trained setting.

In short: **fine-tuning the last block is the best trade-off** - near-full accuracy at a fraction of the cost; head-only if compute is critical; full fine-tune only if accuracy is paramount and time is available.

### Which layers seem most transferable, and why?

- **Early layers transfer almost for free.** They encode generic primitives (edges, corners, colors, textures) that are identical across datasets, so freezing them costs almost nothing - the head-only baseline proves this.
- **Late layers are the most dataset-specific.** They encode semantic parts and object concepts tuned to ImageNet's classes; adapting layer4 (and the head) is where most of the fine-tuning gain comes from.
- This is why fine-tuning only the last block captures most of the full-fine-tune improvement: the top of the hierarchy is the only part that genuinely needs re-learning for a new domain.